In [ ]:
import mne
import matplotlib.pyplot as plt
import os
import numpy as np

subject_ids = ['sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-12']
DATA_PATH = "../Dataset/EEG/" # 사용자 설정 경로
# SUBJECT_ID = 'sub-02'  # 예시 피험자
TASK_NAME = 'task-MIvsRest' # ds003810 데이터셋의 과제명
RUN_ID = 'run-1' # 예시 run
for SUBJECT_ID in subject_ids:
    # --- 1. 경로 및 피험자 정보 설정 ---
    eeg_file_name = f"{SUBJECT_ID}_{TASK_NAME}_{RUN_ID}_eeg.edf"
    eeg_file_path = os.path.join(DATA_PATH, SUBJECT_ID, 'eeg', eeg_file_name)

    print(f"로딩할 파일 경로: {eeg_file_path}")

    # --- 2. EEG 데이터 로드 ---
    try:
        # preload=True로 설정하여 데이터를 메모리에 미리 로드 (전처리 시 필요)
        raw = mne.io.read_raw_edf(eeg_file_path, preload=True, verbose='warning')
        print("EEG 데이터 로드 성공!")
    except FileNotFoundError:
        print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {eeg_file_path}")
        exit()
    except Exception as e:
        print(f"EEG 데이터 로드 중 오류 발생: {e}")
        exit()

    print(f"원본 데이터 채널: {raw.ch_names}")
    print(f"원본 데이터 샘플링 속도: {raw.info['sfreq']} Hz")

    # --- 3. 채널 타입 및 불필요한 채널 처리 (필요시) ---
    # 만약 로드된 데이터에 EEG가 아닌 채널(EMG, STIM 등)이 있다면, 여기서 EEG 채널만 선택합니다.
    # 현재 로드된 채널들이 ('Pz', 'Cz', ...) 모두 EEG 채널이라고 가정합니다.
    # 만약 특정 채널을 제외하고 싶다면 raw.drop_channels(['channel_name']) 사용 가능.
    # 현재는 모든 채널을 EEG로 간주하고 진행합니다.
    # raw.set_channel_types({'STIM': 'stim'}) # 만약 STIM 채널이 있다면 타입 지정

    # --- 4. 전원선 노이즈 제거 (Power Line Noise Removal) ---
    # 데이터셋 출처(아르헨티나 등)를 고려하여 50Hz로 설정
    line_freq = 50.0
    # 데이터 샘플링 주파수가 125Hz이므로, 50Hz의 1차 고조파(100Hz)는 Nyquist 주파수(62.5Hz)를 초과.
    # 따라서 50Hz 노이즈만 제거.
    notch_freqs = [line_freq]
    raw.notch_filter(freqs=notch_freqs, fir_design='firwin', trans_bandwidth=1.0, verbose='warning')
    print(f"\n{notch_freqs} Hz 에서의 전원선 노이즈 제거 완료.")

    # --- 5. 대역 통과 필터링 (Band-pass Filtering) ---
    # Readme.md 정보 기반: 0.5 - 45 Hz, 3rd order Butterworth
    l_freq_butter, h_freq_butter = 0.5, 45.0
    filter_order = 3 # 3rd order Butterworth

    # IIR 필터 파라미터 설정
    iir_params = dict(order=filter_order, ftype='butter', output='sos') # 'sos'는 안정성을 위해 권장

    # method='iir'로 설정하고 iir_params 전달
    raw.filter(l_freq=l_freq_butter, h_freq=h_freq_butter, method='iir', 
            iir_params=iir_params, skip_by_annotation='edge', verbose='warning')

    print(f"\n{l_freq_butter}-{h_freq_butter} Hz, {filter_order}차 Butterworth 대역 통과 필터링 완료.")
    
    # --- 6. 채널 재참조 (Re-referencing) ---
    # 평균 레퍼런스 적용
    raw.set_eeg_reference(ref_channels='average', projection=True, verbose='warning')
    print("\n평균 레퍼런스 적용 완료.")

    # --- 7. ICA (Independent Component Analysis) 기반 아티팩트 제거 ---
    print("\n--- ICA (Independent Component Analysis) 수행 ---")
    # n_components: 설명된 분산(0.95) 또는 채널 수에 가까운 정수값 고려 가능.
    # 15개 채널이므로, 0.95는 약 14개 컴포넌트, 또는 정수로 14 (n_channels - 1) 등을 사용.
    # rank 문제 방지를 위해 n_components를 채널 수보다 작게 설정하는 것이 일반적.
    n_ica_components = min(0.95, len(raw.ch_names) -1) # rank 고려
    ica = mne.preprocessing.ICA(n_components=n_ica_components, random_state=99, method='fastica', max_iter='auto')
    ica.fit(raw)

    # EOG 아티팩트 자동 식별
    # 현재 raw 객체의 채널 목록 ('Pz', 'Cz', ..., 'Fz', ...)에 'Fp1', 'Fp2'가 없으므로 'Fz' 사용
    eog_ch_for_ica = 'Fz'
    if eog_ch_for_ica in raw.ch_names:
        eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name=eog_ch_for_ica)
        if eog_indices:
            ica.exclude.extend(eog_indices)
        print(f"\n자동으로 식별된 EOG 아티팩트 컴포넌트 (기준 채널 '{eog_ch_for_ica}'): {eog_indices}")
    else:
        print(f"\n경고: EOG 아티팩트 자동 식별을 위한 채널 '{eog_ch_for_ica}'를 찾을 수 없습니다.")
        eog_indices = []


    # 근육 잡음 아티팩트 자동 식별
    # find_bads_muscle은 고주파 필터링된 데이터에서 더 잘 작동할 수 있음
    # raw_for_muscle_ica = raw.copy().filter(l_freq=20, h_freq=None) # 예시
    # muscle_indices, muscle_scores = ica.find_bads_muscle(raw_for_muscle_ica)
    muscle_indices, muscle_scores = ica.find_bads_muscle(raw)
    if muscle_indices:
        ica.exclude.extend(muscle_indices)
    print(f"자동으로 식별된 근육 아티팩트 컴포넌트: {muscle_indices}")

    # 중복 제거 및 정렬
    ica.exclude = sorted(list(set(ica.exclude)))
    print(f"\n최종적으로 제외할 ICA 컴포넌트: {ica.exclude}")

    # (선택 사항) ICA 컴포넌트 시각적 확인 후 수동 조정 단계
    # print("ICA 컴포넌트 시각화: 제외할 컴포넌트를 확인하고 필요한 경우 ica.exclude 리스트를 수정하세요.")
    # ica.plot_components(picks=range(ica.n_components_))
    # plt.show(block=True) # 사용자가 창을 닫을 때까지 대기
    # # 예시: ica.exclude = [0, 3] # 시각적 확인 후 직접 설정

    # ica.plot_sources(raw, show_scrollbars=True, block=True) # 원본 데이터에 적용된 컴포넌트 확인
    # plt.show(block=True)
    # # 예시: ica.exclude = [1, 4] # 시각적 확인 후 직접 설정

    # 아티팩트 컴포넌트 제거 및 원본 데이터에 적용
    if ica.exclude:
        ica.apply(raw)
        print("\nICA 기반 아티팩트 제거 완료.")
    else:
        print("\n제외할 ICA 컴포넌트가 선택되지 않아 ICA 적용을 건너 <0xEB><0x9B><0x84>니다.")


    # --- 8. 전처리 완료된 데이터 확인 및 추출 ---
    print("\n--- 모든 전처리 완료 ---")
    # 전처리된 데이터 시각화 (선택 사항)
    # raw.plot(duration=10, n_channels=15, scalings='auto', title='Preprocessed Continuous EEG Data')
    # plt.show(block=True)

    # 전처리된 EEG 데이터를 NumPy 배열로 추출
    cleaned_eeg_data = raw.get_data() # (n_channels, n_samples)
    print(f"\n전처리 완료된 EEG 데이터 shape: {cleaned_eeg_data.shape}")
    print("이 cleaned_eeg_data 변수를 사용하여 다음 분석을 진행할 수 있습니다.")

    # (선택 사항) 전처리된 raw 객체 저장
    output_dir = os.path.join(DATA_PATH, SUBJECT_ID, 'derivatives') # 예시 출력 폴더
    os.makedirs(output_dir, exist_ok=True)
    preprocessed_file_name = f"{SUBJECT_ID}_{TASK_NAME}_{RUN_ID}_preprocessed_raw.fif"
    raw.save(os.path.join(output_dir, preprocessed_file_name), overwrite=True)
    print(f"\n전처리된 데이터가 다음 경로에 저장되었습니다: {os.path.join(output_dir, preprocessed_file_name)}")

In [ ]:
import mne
import matplotlib.pyplot as plt
import os
import numpy as np

In [ ]:
# --- 1. 경로 및 피험자 정보 설정 ---
DATA_PATH = "../Dataset/EEG/" # 사용자 설정 경로
SUBJECT_ID = 'sub-05'  # 예시 피험자
TASK_NAME = 'task-MIvsRest' # ds003810 데이터셋의 과제명
RUN_ID = 'run-3' # 예시 run

# 파일 경로 구성
eeg_file_name = f"{SUBJECT_ID}_{TASK_NAME}_{RUN_ID}_eeg.edf"
eeg_file_path = os.path.join(DATA_PATH, SUBJECT_ID, 'eeg', eeg_file_name)

print(f"로딩할 파일 경로: {eeg_file_path}")

In [ ]:
# --- 2. EEG 데이터 로드 ---
try:
    # preload=True로 설정하여 데이터를 메모리에 미리 로드 (전처리 시 필요)
    raw = mne.io.read_raw_edf(eeg_file_path, preload=True, verbose='warning')
    print("EEG 데이터 로드 성공!")
except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {eeg_file_path}")
    exit()
except Exception as e:
    print(f"EEG 데이터 로드 중 오류 발생: {e}")
    exit()

print(f"원본 데이터 채널: {raw.ch_names}")
print(f"원본 데이터 샘플링 속도: {raw.info['sfreq']} Hz")

In [ ]:
# --- 4. 전원선 노이즈 제거 (Power Line Noise Removal) ---
# 데이터셋 출처(아르헨티나 등)를 고려하여 50Hz로 설정
line_freq = 50.0
# 데이터 샘플링 주파수가 125Hz이므로, 50Hz의 1차 고조파(100Hz)는 Nyquist 주파수(62.5Hz)를 초과.
# 따라서 50Hz 노이즈만 제거.
notch_freqs = [line_freq]
raw.notch_filter(freqs=notch_freqs, fir_design='firwin', trans_bandwidth=1.0, verbose='warning')
print(f"\n{notch_freqs} Hz 에서의 전원선 노이즈 제거 완료.")

In [ ]:
# --- 5. 대역 통과 필터링 (Band-pass Filtering) ---
# Readme.md 정보 기반: 0.5 - 45 Hz, 3rd order Butterworth
l_freq_butter, h_freq_butter = 0.5, 45.0
filter_order = 3 # 3rd order Butterworth

# IIR 필터 파라미터 설정
iir_params = dict(order=filter_order, ftype='butter', output='sos') # 'sos'는 안정성을 위해 권장

# method='iir'로 설정하고 iir_params 전달
raw.filter(l_freq=l_freq_butter, h_freq=h_freq_butter, method='iir', 
           iir_params=iir_params, skip_by_annotation='edge', verbose='warning')

print(f"\n{l_freq_butter}-{h_freq_butter} Hz, {filter_order}차 Butterworth 대역 통과 필터링 완료.")

In [ ]:
# --- 6. 채널 재참조 (Re-referencing) ---
# 평균 레퍼런스 적용
raw.set_eeg_reference(ref_channels='average', projection=True, verbose='warning')
print("\n평균 레퍼런스 적용 완료.")

In [ ]:
# --- 7. ICA (Independent Component Analysis) 기반 아티팩트 제거 ---
print("\n--- ICA (Independent Component Analysis) 수행 ---")
# n_components: 설명된 분산(0.95) 또는 채널 수에 가까운 정수값 고려 가능.
# 15개 채널이므로, 0.95는 약 14개 컴포넌트, 또는 정수로 14 (n_channels - 1) 등을 사용.
# rank 문제 방지를 위해 n_components를 채널 수보다 작게 설정하는 것이 일반적.
n_ica_components = min(0.95, len(raw.ch_names) -1) # rank 고려
ica = mne.preprocessing.ICA(n_components=n_ica_components, random_state=99, method='fastica', max_iter='auto')
ica.fit(raw)

In [ ]:
# EOG 아티팩트 자동 식별
# 현재 raw 객체의 채널 목록 ('Pz', 'Cz', ..., 'Fz', ...)에 'Fp1', 'Fp2'가 없으므로 'Fz' 사용
eog_ch_for_ica = 'Fz'
if eog_ch_for_ica in raw.ch_names:
    eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name=eog_ch_for_ica)
    if eog_indices:
        ica.exclude.extend(eog_indices)
    print(f"\n자동으로 식별된 EOG 아티팩트 컴포넌트 (기준 채널 '{eog_ch_for_ica}'): {eog_indices}")
else:
    print(f"\n경고: EOG 아티팩트 자동 식별을 위한 채널 '{eog_ch_for_ica}'를 찾을 수 없습니다.")
    eog_indices = []

In [ ]:
# 근육 잡음 아티팩트 자동 식별
# find_bads_muscle은 고주파 필터링된 데이터에서 더 잘 작동할 수 있음
# raw_for_muscle_ica = raw.copy().filter(l_freq=20, h_freq=None) # 예시
# muscle_indices, muscle_scores = ica.find_bads_muscle(raw_for_muscle_ica)
muscle_indices, muscle_scores = ica.find_bads_muscle(raw)
if muscle_indices:
    ica.exclude.extend(muscle_indices)
print(f"자동으로 식별된 근육 아티팩트 컴포넌트: {muscle_indices}")

In [ ]:
# 중복 제거 및 정렬
ica.exclude = sorted(list(set(ica.exclude)))
print(f"\n최종적으로 제외할 ICA 컴포넌트: {ica.exclude}")

In [ ]:
# 아티팩트 컴포넌트 제거 및 원본 데이터에 적용
if ica.exclude:
    ica.apply(raw)
    print("\nICA 기반 아티팩트 제거 완료.")
else:
    print("\n제외할 ICA 컴포넌트가 선택되지 않아 ICA 적용을 건너 <0xEB><0x9B><0x84>니다.")

In [ ]:
# --- 8. 전처리 완료된 데이터 확인 및 추출 ---
print("\n--- 모든 전처리 완료 ---")
# 전처리된 데이터 시각화 (선택 사항)
# raw.plot(duration=10, n_channels=15, scalings='auto', title='Preprocessed Continuous EEG Data')
# plt.show(block=True)

# 전처리된 EEG 데이터를 NumPy 배열로 추출
cleaned_eeg_data = raw.get_data() # (n_channels, n_samples)
print(f"\n전처리 완료된 EEG 데이터 shape: {cleaned_eeg_data.shape}")
print("이 cleaned_eeg_data 변수를 사용하여 다음 분석을 진행할 수 있습니다.")

In [ ]:
# 전처리된 raw 객체 저장
output_dir = os.path.join(DATA_PATH, 'derivatives') # 출력 폴더
os.makedirs(output_dir, exist_ok=True)
preprocessed_file_name = f"{SUBJECT_ID}_{TASK_NAME}_{RUN_ID}_preprocessed_raw.fif"
raw.save(os.path.join(output_dir, preprocessed_file_name), overwrite=True)
print(f"\n전처리된 데이터가 다음 경로에 저장되었습니다: {os.path.join(output_dir, preprocessed_file_name)}")